# Ejercicio Módulo 2
**Inteligencia Artificial - CEIA - FIUBA**

**Matías Agustín Souto**

En este ejercicio deben implementar un algoritmo de búsqueda que no sea **Búsqueda Primero en Anchura (BFS)** para resolver el problema de la Torre de Hanoi. La nota máxima dependerá del algoritmo implementado:

- **Búsqueda Primero en Profundidad**: nota máxima 6.
- **Búsqueda de Costo Uniforme**: nota máxima 6.
- **Búsqueda de Profundidad Limitada con Profundidad Iterativa**: nota máxima 7.
- **Búsqueda Voraz usando la heurística dada en el aula virtual**: nota máxima 8.
- **Búsqueda Voraz usando una heurística desarrollada por vos**: nota máxima 9.
- **Búsqueda A\* usando la heurística dada en el aula virtual**: nota máxima 9.
- **Búsqueda A\* usando una heurística desarrollada por vos**: nota máxima 10.

La función debe devolver la salida correspondiente a la solución encontrada o `None si no se encontró una solución.

Además, debe calcular métricas de rendimiento que, como mínimo, incluyan:

- `solution_found`: `True` si se encontró la solución, `False` en caso contrario.
- `nodes_explored`: cantidad de nodos explorados (entero).
- `states_visited`: cantidad de estados distintos visitados (entero).
- `nodes_in_frontier`: cantidad de nodos que quedaron en la frontera al finalizar la ejecución (entero).
- `max_depth`: máxima profundidad explorada (entero).
- `cost_total`: costo total para encontrar la solución (float).

In [66]:
from aima_libs.hanoi_states import ProblemHanoi, StatesHanoi
from aima_libs.tree_hanoi import NodeHanoi

In [72]:
def search_algorithm(number_disks=5) -> (NodeHanoi, dict):
    def optimal_moves_count():
        """Segun las demostraciones que se citan a continuación, esta es la solucion
        óptima para este problema con n discos:
        - https://www.jdhwilkins.com/proof-by-induction-ft-the-tower-of-hanoi
        - https://www.wiris.com/es/blog/torres-de-hanoi-un-desafio-de-matematicas-y-programacion/
        """
        return 2**number_disks -1
    def a_star_heuristic(node, possible_states):
        goal = possible_states.get_state()[2]
        node_value = 0
        target_rod_status = node.state.get_state()[2]
        for i in range(len(goal)):
            if len(target_rod_status) > i:
                if target_rod_status[i] == goal[i]:
                    node_value += 1
            else:
                break
        #Para estimar el valor de heurística, se toma la diferencia entre
        #los discos esperados y la cantidad de discos en buena posición
        #Se agrega también un factor que lo multiplica por el valor óptimo del problema
        #para reforzar más el costo de lo que está en una mala posición. El factor viene dado
        #por conocimiento de negocio
        return (len(goal) - node_value) * optimal_moves_count()
    
    def get_metrics(**kwargs): 
        return dict(kwargs)

    list_disks = [i for i in range(number_disks, 0, -1)]
    initial_state = StatesHanoi(list_disks, [], [], max_disks=number_disks)
    goal_state = StatesHanoi([], [], list_disks, max_disks=number_disks)
    problem = ProblemHanoi(initial=initial_state, goal=goal_state)

    from queue import PriorityQueue

    queue = PriorityQueue()
    queue.put((0, NodeHanoi(initial_state)))
    explored_set = set()
    nodes_explored = 0
    max_depth_reached = 0

    while queue.qsize() > 0:
        cost, current_node = queue.get()
        nodes_explored += 1
        explored_set.add(current_node.state)
        if (max_depth := current_node.depth)  > max_depth_reached:
            max_depth_reached = max_depth
        if problem.goal_test(current_node.state):
            metrics = get_metrics(
                solution_found = True,
                nodes_explored = nodes_explored,
                states_visited = len(explored_set),
                nodes_in_frontier = queue.qsize(),
                max_depth = max_depth_reached,
                cost_total = current_node.path_cost,
                optimal_solution_size = optimal_moves_count()
            )
            return current_node, metrics
        for next_node in current_node.expand(problem):
            if next_node.state not in explored_set:
                queue.put(
                    (next_node.path_cost + a_star_heuristic(next_node,problem.goal), next_node)
                )

    metrics = get_metrics(
                solution_found = False,
                nodes_explored = nodes_explored,
                states_visited = len(explored_set),
                nodes_in_frontier = queue.qsize(),
                max_depth = max_depth_reached,
                cost_total = current_node.path_cost,
                optimal_solution_size = optimal_moves_count()

            )
    return None, metrics


Se prueba la función:

In [73]:
solution, metrics = search_algorithm(number_disks=10)

Veamos las métricas:

In [74]:
for key, value in metrics.items():
    print(f"{key}: {value}")

solution_found: True
nodes_explored: 60365
states_visited: 29076
nodes_in_frontier: 1800
max_depth: 1023
cost_total: 1023.0
optimal_solution_size: 1023


Veamos el camino de estados desde el principio a la solución:

In [70]:
for nodos in solution.path():
    print(nodos)

<Node HanoiState: 10 9 8 7 6 5 4 3 2 1 |  | >
<Node HanoiState: 10 9 8 7 6 5 4 3 2 | 1 | >
<Node HanoiState: 10 9 8 7 6 5 4 3 | 1 | 2>
<Node HanoiState: 10 9 8 7 6 5 4 3 |  | 2 1>
<Node HanoiState: 10 9 8 7 6 5 4 | 3 | 2 1>
<Node HanoiState: 10 9 8 7 6 5 4 1 | 3 | 2>
<Node HanoiState: 10 9 8 7 6 5 4 1 | 3 2 | >
<Node HanoiState: 10 9 8 7 6 5 4 | 3 2 1 | >
<Node HanoiState: 10 9 8 7 6 5 | 3 2 1 | 4>
<Node HanoiState: 10 9 8 7 6 5 | 3 2 | 4 1>
<Node HanoiState: 10 9 8 7 6 5 2 | 3 | 4 1>
<Node HanoiState: 10 9 8 7 6 5 2 1 | 3 | 4>
<Node HanoiState: 10 9 8 7 6 5 2 1 |  | 4 3>
<Node HanoiState: 10 9 8 7 6 5 2 | 1 | 4 3>
<Node HanoiState: 10 9 8 7 6 5 | 1 | 4 3 2>
<Node HanoiState: 10 9 8 7 6 5 |  | 4 3 2 1>
<Node HanoiState: 10 9 8 7 6 | 5 | 4 3 2 1>
<Node HanoiState: 10 9 8 7 6 1 | 5 | 4 3 2>
<Node HanoiState: 10 9 8 7 6 1 | 5 2 | 4 3>
<Node HanoiState: 10 9 8 7 6 | 5 2 1 | 4 3>
<Node HanoiState: 10 9 8 7 6 3 | 5 2 1 | 4>
<Node HanoiState: 10 9 8 7 6 3 | 5 2 | 4 1>
<Node HanoiState: 10 9 8

Y las acciones que el agente debería aplicar para llegar al objetivo:

In [71]:
for act in solution.solution():
    print(act)

Move disk 1 from 1 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 3 from 1 to 2
Move disk 1 from 3 to 1
Move disk 2 from 3 to 2
Move disk 1 from 1 to 2
Move disk 4 from 1 to 3
Move disk 1 from 2 to 3
Move disk 2 from 2 to 1
Move disk 1 from 3 to 1
Move disk 3 from 2 to 3
Move disk 1 from 1 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 5 from 1 to 2
Move disk 1 from 3 to 1
Move disk 2 from 3 to 2
Move disk 1 from 1 to 2
Move disk 3 from 3 to 1
Move disk 1 from 2 to 3
Move disk 2 from 2 to 1
Move disk 1 from 3 to 1
Move disk 4 from 3 to 2
Move disk 1 from 1 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 3 from 1 to 2
Move disk 1 from 3 to 1
Move disk 2 from 3 to 2
Move disk 1 from 1 to 2
Move disk 6 from 1 to 3
Move disk 1 from 2 to 3
Move disk 2 from 2 to 1
Move disk 1 from 3 to 1
Move disk 3 from 2 to 3
Move disk 1 from 1 to 2
Move disk 2 from 1 to 3
Move disk 1 from 2 to 3
Move disk 4 from 2 to 1
Move disk 1 from 3 to 1
Move disk 2 from